## **Entrenamiento del modelo con datos sin limpieza previa**

* Concluido el análisis exploratorio de datos (EDA), en esta segunda etapa del proyecto se procederá a entrenar un modelo predictivo utilizando los datos en su estado original, es decir, sin aplicar procesos de limpieza ni tratamiento avanzado de los datos. Únicamente se realizará un preprocesamiento básico, que incluye la estandarización de las variables numéricas mediante StandardScaler y la codificación de las variables categóricas a través de One-Hot Encoding.

* El objetivo de este experimento es establecer una línea base de desempeño, que permita evaluar el impacto que tiene la calidad de los datos sobre el rendimiento del modelo. Posteriormente, los resultados obtenidos serán comparados con los de un segundo modelo entrenado sobre datos debidamente limpiados y tratados, con el fin de evidenciar las mejoras en rendimiento, estabilidad e interpretabilidad.

* Este enfoque permitirá demostrar que la correcta preparación de los datos es un componente fundamental en proyectos de ciencia de datos, y que incluso con el mismo algoritmo, la calidad del preprocesamiento puede marcar una diferencia significativa en los resultados obtenidos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/dataset/car_sales_data.csv')

In [ ]:
df.head()

,Fabricante,Modelo,Motor,Combustible,Año,Kilometraje,Precio
0,Ford,Fiesta,1.0,Petrol,2002,127300,3074
1,Porsche,718 Cayman,4.0,Petrol,2016,57850,49704
2,Ford,Mondeo,1.6,Diesel,2014,39190,24072
3,Toyota,RAV4,1.8,Hybrid,1988,210814,1705
4,VW,Polo,1.0,Petrol,2006,127869,4101


## **Separación de los datos en entrenamiento, validación y prueba**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = df.drop('Precio', axis=1)
y = df['Precio']

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train,X_val,y_train,y_val = train_test_split(X_train,y_train,test_size=0.25,random_state=42)

* Para dividir el conjunto de datos en entrenamiento, validación y prueba, se utiliza la función train_test_split de la librería scikit-learn. Esta separación se realiza con el objetivo de evaluar la capacidad de generalización del modelo y prevenir el sobreajuste (overfitting).

* El modelo se entrena exclusivamente con el conjunto de entrenamiento, mientras que el conjunto de validación se emplea para ajustar hiperparámetros y tomar decisiones durante el proceso de modelado. Finalmente, el conjunto de prueba se utiliza únicamente para evaluar el desempeño final del modelo sobre datos no vistos.

* Este procedimiento permite verificar si el modelo es capaz de aprender patrones generales a partir de los datos, en lugar de memorizar el conjunto de entrenamiento, lo cual es fundamental para garantizar un buen desempeño en escenarios reales.

## **Preprocesamiento de los datos**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [ ]:
num_features = [ 'Kilometraje', 'Motor','Año']
cat_features = ['Modelo','Fabricante', 'Combustible']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ]
)

* En la etapa de preprocesamiento se aplican distintas transformaciones según el tipo de variable. Para las variables numéricas se utiliza **StandardScaler**, el cual estandariza los datos restando el valor promedio y dividiendo entre la desviación estándar. Este proceso permite que todas las variables numéricas queden en una misma escala, evitando que aquellas con magnitudes mayores dominen el entrenamiento del modelo y facilitando la comparación entre ellas.

* Por otro lado, para las variables categóricas nominales se emplea la técnica de **One-Hot Encoding**, ya que este tipo de variables no posee un orden intrínseco. Esta transformación consiste en crear variables ficticias (dummies) representadas por valores binarios, donde se asigna un 1 a la categoría correspondiente y 0 a las demás, permitiendo que el modelo pueda interpretar correctamente la información categórica.

* Adicionalmente, se hace uso de funciones como ColumnTransformer, la cual permite asignar de forma explícita el preprocesador adecuado a cada conjunto de variables según su naturaleza (numéricas o categóricas). Para ello, se especifica el tipo de transformación a aplicar junto con el nombre de las variables correspondientes.

* Esta estrategia resulta especialmente útil cuando el modelo se lleva a producción, ya que garantiza que los datos nuevos reciban exactamente el mismo tratamiento que los datos utilizados durante el entrenamiento. Además, mejora la reproducibilidad, reduce errores manuales y simplifica el flujo de trabajo al integrar todo el preprocesamiento dentro del pipeline del modelo.

## **Creación del modelo**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', LinearRegression())
])

In [ ]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Kilometraje', 'Motor',
                                                   'Año']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Modelo', 'Fabricante',
                                                   'Combustible'])])),
                ('model', LinearRegression())])

* Mediante el uso de la clase Pipeline se integra un modelo de regresión lineal múltiple, el cual corresponde a un algoritmo estadístico cuyo objetivo es encontrar la relación lineal que mejor se ajusta a los datos.

* En el caso de la regresión lineal simple, el modelo busca encontrar la mejor recta que describe la relación entre una única variable independiente y la variable objetivo. Sin embargo, en este proyecto se emplea una regresión lineal múltiple, ya que el precio del vehículo depende de varias variables explicativas, tales como el fabricante, modelo, tipo de motor, tipo de combustible, año de fabricación y kilometraje.

* En este proyecto se utiliza la función ColumnTransformer para aplicar de manera diferenciada las transformaciones necesarias a cada tipo de variable. Las variables numéricas, como el año de fabricación, el kilometraje y la cilindrada del motor, requieren procesos como el escalamiento, mientras que las variables categóricas, como fabricante, modelo y tipo de combustible, necesitan técnicas de codificación adecuadas. El uso de ColumnTransformer permite centralizar este preprocesamiento dentro de una única estructura, garantizando consistencia y evitando fugas de información.

* En este contexto, el modelo no ajusta una recta, sino un hiperplano en un espacio de múltiples dimensiones, el cual representa la combinación lineal de todas las variables independientes que mejor explica el comportamiento del precio del vehículo.

* Si bien la regresión lineal asume una relación lineal entre las variables, puede utilizarse en escenarios donde la tendencia no es estrictamente lineal, siempre y cuando se realice un preprocesamiento adecuado, como transformaciones de variables, codificación correcta de variables categóricas y escalamiento, aspectos que se abordarán en etapas posteriores del proyecto.

## **Métricas**

In [ ]:
pipeline.score(X_train,y_train)

0.7137528004450833

In [ ]:
pipeline.score(X_test,y_test)

0.7101567973982701

In [ ]:
pipeline.score(X_val,y_val)

0.7230949962840827

* Muchos algoritmos de scikit-learn incorporan una función score, que permite evaluar el desempeño del modelo de forma directa. En el caso de los modelos de regresión, como la regresión lineal múltiple, el método score devuelve el coeficiente de determinación **$R^2$** , el cual mide la proporción de la varianza de la variable objetivo que es explicada por el modelo.



* Un valor de **$R^2$** más alto indica un mayor poder explicativo, ya que una mayor parte de la variabilidad observada en los valores reales es capturada por las predicciones del modelo.




In [ ]:
from sklearn.metrics import mean_absolute_error

In [ ]:
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)
y_val_pred = pipeline.predict(X_val)

In [ ]:
print('MAE train: ',mean_absolute_error(y_train,y_train_pred))
print('MAE test: ',mean_absolute_error(y_test,y_test_pred))
print('MAE val: ',mean_absolute_error(y_val,y_val_pred))

MAE train:  5779.990333539248
MAE test:  5781.497122533276
MAE val:  5678.012854705562


* El error absoluto medio es una métrica más robusta frente a outliers, ya que penaliza los errores de forma lineal, mientras que el error cuadráditico medio son menos tolerantes a errores de gran magnitud debido a la penalización cuadrática.